Testes de Validação dos Dados

In [10]:
import pandas as pd
import numpy as np
from datetime import datetime

dim_cliente = pd.read_csv("../Dados Finais/dim_cliente.csv",  encoding="utf-8-sig")
dim_instrutor = pd.read_csv("../Dados Finais/dim_instrutor.csv",  encoding="utf-8-sig")
dim_campanha = pd.read_csv("../Dados Finais/dim_campanha.csv",  encoding="utf-8-sig")
dim_treino = pd.read_csv("../Dados Finais/dim_treino.csv",  encoding="utf-8-sig")
dim_aula = pd.read_csv("../Dados Finais/dim_aula.csv",  encoding="utf-8-sig")

tf_cliente= pd.read_csv("../Dados Finais/tf_cliente.csv",  encoding="utf-8-sig")
tf_instrutor= pd.read_csv("../Dados Finais/tf_instrutor.csv",  encoding="utf-8-sig")

Testes de Integridade Referencial

In [11]:
def valida_referencial(nome, chaves_facto, chaves_dim, label):
    erros = set(chaves_facto) - set(chaves_dim)
    if erros:
        print(f"[ERRO] {nome}: Existem {label} órfãs na tabela de factos que não existem na dimensão: {erros}")
    else:
        print(f"[OK] {nome}: Todas as {label} da tabela de factos existem na dimensão.")

# tf_cliente
valida_referencial('tf_cliente', tf_cliente['sk_cliente'], dim_cliente['cliente_sk'], 'sk_cliente')
valida_referencial('tf_cliente', tf_cliente['sk_treino'].dropna(), dim_treino['treino_sk'], 'sk_treino')
valida_referencial('tf_cliente', tf_cliente['sk_aula'].dropna(), dim_aula['aula_sk'], 'sk_aula')
valida_referencial('tf_cliente', tf_cliente['sk_campanha'].dropna(), dim_campanha['campanha_sk'], 'sk_campanha')

# tf_instrutor
valida_referencial('tf_instrutor', tf_instrutor['sk_instrutor'], dim_instrutor['instrutor_sk'], 'sk_instrutor')
valida_referencial('tf_instrutor', tf_instrutor['sk_aula'], dim_aula['aula_sk'], 'sk_aula')

[OK] tf_cliente: Todas as sk_cliente da tabela de factos existem na dimensão.
[OK] tf_cliente: Todas as sk_treino da tabela de factos existem na dimensão.
[OK] tf_cliente: Todas as sk_aula da tabela de factos existem na dimensão.
[OK] tf_cliente: Todas as sk_campanha da tabela de factos existem na dimensão.
[OK] tf_instrutor: Todas as sk_instrutor da tabela de factos existem na dimensão.
[OK] tf_instrutor: Todas as sk_aula da tabela de factos existem na dimensão.


Testes de Integridade de Negócio

In [12]:
# 1. Cada cliente faz pelo menos uma atividade (aula ou treino) em cada ida
sem_atividade = ((tf_cliente['sk_treino'].isna()) & (tf_cliente['sk_aula'].isna())).sum()
if sem_atividade == 0:
    print("[OK] Todas as idas têm pelo menos uma atividade (aula ou treino).")
else:
    print(f"[ERRO] Existem {sem_atividade} idas sem qualquer atividade!")

# 2. Avaliações apenas ocorrem quando o cliente faz essas atividades
avaliacao_treino_sem_treino = tf_cliente[(tf_cliente['avaliacao_treino'].notna()) & (tf_cliente['sk_treino'].isna())].shape[0]
avaliacao_aula_sem_aula = tf_cliente[(tf_cliente['avaliacao_aula'].notna()) & (tf_cliente['sk_aula'].isna())].shape[0]
if avaliacao_treino_sem_treino == 0:
    print("[OK] Todas as avaliações de treino ocorrem apenas quando há treino.")
else:
    print(f"[ERRO] Existem {avaliacao_treino_sem_treino} avaliações de treino sem treino associado!")
if avaliacao_aula_sem_aula == 0:
    print("[OK] Todas as avaliações de aula ocorrem apenas quando há aula.")
else:
    print(f"[ERRO] Existem {avaliacao_aula_sem_aula} avaliações de aula sem aula associada!")

# 3. Não existem aulas em dias errados (cada aula só ocorre no seu dia fixo da semana)
mapa_aula_dia = {
    1: 0,  # Alongamentos - segunda
    2: 0,  # Boxe - segunda
    3: 1,  # Cardio - terça
    4: 1,  # Crossfit - terça
    5: 2,  # Funcional - quarta
    6: 2,  # HIIT - quarta
    7: 4,  # Musculação - sexta
    8: 4,  # Pilates - sexta
    9: 3,  # Yoga - quinta
    10: 3  # Zumba - quinta
}
tf_cliente['data'] = pd.to_datetime(tf_cliente['data'])
tf_cliente['dia_semana'] = tf_cliente['data'].dt.weekday
aulas_invalidas = []
for idx, row in tf_cliente[tf_cliente['sk_aula'].notna()].iterrows():
    aula_id = int(dim_aula.loc[dim_aula['aula_sk'] == row['sk_aula'], 'aula_id'].values[0])
    dia_correto = mapa_aula_dia.get(aula_id, None)
    if dia_correto is not None and row['dia_semana'] != dia_correto:
        aulas_invalidas.append(idx)
if len(aulas_invalidas) == 0:
    print("[OK] Não existem aulas em dias errados.")
else:
    print(f"[ERRO] Existem {len(aulas_invalidas)} idas a aulas em dias errados.")

# 4. Cada instrutor tem uma aula de que está encarregue
aulas_por_instrutor = dim_aula['aula_sk'].nunique()
instrutores = dim_instrutor['instrutor_sk'].nunique()
if aulas_por_instrutor == instrutores:
    print("[OK] Cada instrutor tem uma aula atribuída (1:1).")
else:
    print(f"[ERRO] Existem {instrutores - aulas_por_instrutor} instrutores sem aula atribuída ou vice-versa.")

# 5. Cada instrutor tem pelo menos um plano associado a si
instrutores_com_plano = dim_treino['instrutor_nome'].nunique()
instrutores_total = dim_instrutor['instrutor_nome'].nunique()
if instrutores_com_plano == instrutores_total:
    print("[OK] Todos os instrutores têm pelo menos um plano associado.")
else:
    print(f"[ERRO] Existem {instrutores_total - instrutores_com_plano} instrutores sem plano associado!")

[OK] Todas as idas têm pelo menos uma atividade (aula ou treino).
[OK] Todas as avaliações de treino ocorrem apenas quando há treino.
[OK] Todas as avaliações de aula ocorrem apenas quando há aula.
[OK] Não existem aulas em dias errados.
[OK] Cada instrutor tem uma aula atribuída (1:1).
[OK] Todos os instrutores têm pelo menos um plano associado.


Testes de Plausibilidade

In [13]:
# 1. Não existem horários fora do intervalo permitido
horarios_invalidos = tf_cliente[(tf_cliente['hora_entrada'] < 9) | (tf_cliente['hora_entrada'] > 20) | (tf_cliente['hora_saida'] > 22)]
if horarios_invalidos.empty:
    print("[OK] Todos os horários de entrada e saída estão dentro do intervalo permitido.")
else:
    print(f"[ERRO] Existem {horarios_invalidos.shape[0]} idas com horários fora do intervalo permitido!")

# 2. Não existem avaliações fora do intervalo 1-5 (se aplicável)
avaliacoes_invalidas = tf_cliente[
    ((tf_cliente['avaliacao_treino'].notna()) & ((tf_cliente['avaliacao_treino'] < 1) | (tf_cliente['avaliacao_treino'] > 5))) |
    ((tf_cliente['avaliacao_aula'].notna()) & ((tf_cliente['avaliacao_aula'] < 1) | (tf_cliente['avaliacao_aula'] > 5)))
]
if avaliacoes_invalidas.empty:
    print("[OK] Todas as avaliações estão no intervalo 1-5.")
else:
    print(f"[ERRO] Existem {avaliacoes_invalidas.shape[0]} avaliações fora do intervalo 1-5!")

# 3. Não existem duplicados inesperados (mesmo cliente, data, treino, aula)
duplicados = tf_cliente.duplicated(subset=['sk_cliente', 'data', 'sk_treino', 'sk_aula'], keep=False)
if not duplicados.any():
    print("[OK] Não existem duplicados inesperados (cliente, data, treino, aula).")
else:
    print(f"[ERRO] Existem {duplicados.sum()} duplicados inesperados (cliente, data, treino, aula)!")

[OK] Todos os horários de entrada e saída estão dentro do intervalo permitido.
[OK] Todas as avaliações estão no intervalo 1-5.
[OK] Não existem duplicados inesperados (cliente, data, treino, aula).
